# Lab 3 — A2A Multi-Agent System with Session-Derived Credentials

> **Mode: DEMO in class** | Time: ~40 min

This lab stitches everything together.

## Architecture

```
          ┌────────────────────────────────────┐
          │          Identity Provider          │
          │  (users, roles, grants, sessions)   │
          └──────────────┬─────────────────────┘
                         │ SessionToken
                         ▼
          ┌────────────────────────────────────┐
          │       CredentialFactory             │
          │  derive MCP tokens + A2A creds      │
          └──────────────┬─────────────────────┘
                         │
         ┌───────────────┼────────────────┐
         │               │                │
         ▼               ▼                ▼
   [Analytics agent] [Inventory agent] [Writer agent]
    (Lab 1 + Skill)   (MCP inventory)   (pure LLM)
         │               │                │
         └───────┬───────┴────────┬───────┘
                 │                │
                 ▼                ▼
         ┌────────────────────────────────┐
         │  Supervisor (LangGraph)        │
         │  1. list user's granted agents │
         │  2. LLM-classify user request  │
         │  3. check agent is allowed     │
         │  4. derive A2A creds for agent │
         │  5. submit task                │
         └────────────────────────────────┘
                          │
                          ▼
                        User
```

## Auth flow

```
Login ─► Session ─┬─► MCP token(analytics-mcp)  ── used by Analytics agent
                  ├─► MCP token(inventory-mcp)  ── used by Inventory agent
                  ├─► A2A api-key (analytics)   ── used by Supervisor -> Analytics
                  ├─► A2A OAuth token (writer)  ── used by Supervisor -> Writer
                  └─► A2A api-key (inventory)   ── used by Supervisor -> Inventory
```

No credential is stored long-term anywhere except the session; revoking the
session drops every downstream token on the next expiry.


## Step 1 — Boot the platform


In [1]:
import os, sys, json, time
from typing import TypedDict
_cwd = os.getcwd()
PROJECT_ROOT = os.path.abspath(os.path.join(_cwd, ".."))
if not os.path.isdir(os.path.join(PROJECT_ROOT, "lib")):
    PROJECT_ROOT = _cwd
sys.path.insert(0, os.path.join(PROJECT_ROOT, "lib"))

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, ".env"))

from data import DB_PATH
if not os.path.exists(DB_PATH):
    sys.path.insert(0, os.path.join(PROJECT_ROOT, "db"))
    from setup_database import create_database
    create_database()

from identity import IdentityProvider, GrantRegistry, CredentialFactory, seed_lab_users
from a2a_framework import (AgentCard, SecurityScheme, A2AAuthProvider,
                            RemoteAgent, ClientAgent, AgentRegistry, TaskStatus)
from agent_builder import (build_analytics_agent, build_inventory_agent,
                            build_writer_agent, reset_builder_state)
from tracing import get_openai_client, get_langchain_handler, flush

reset_builder_state()
idp = IdentityProvider(); grants = GrantRegistry(); seed_lab_users(idp, grants)
a2a_auth = A2AAuthProvider()
cred_factory = CredentialFactory(idp, grants)
cred_factory.register_a2a_provider(a2a_auth)

oai = get_openai_client()
lh = get_langchain_handler()
MODEL = os.getenv("OPENAI_MODEL", "gpt-4.1-nano")
print("Platform ready.")


  [Langfuse] OpenAI client is traced  -> https://cloud.langfuse.com
  [Langfuse] LangChain CallbackHandler enabled.
Platform ready.


## Step 2 — Register 3 Remote Agents (each with its own Agent Card)

Each Remote Agent advertises its security scheme so supervisors know how to
authenticate.


In [2]:
# Agent cards + handlers ---------------------------------------------------
analytics_card = AgentCard(
    name="analytics_agent",
    description="Queries revenue, runs SQL, applies KPI Skill for reports.",
    endpoint="agent://analytics",
    capabilities=["kpi_report","revenue_query","data_analysis"],
    security=SecurityScheme(scheme_type="apiKey"))

inventory_card = AgentCard(
    name="inventory_agent",
    description="Product stock analysis, flags LOW/CRITICAL items.",
    endpoint="agent://inventory",
    capabilities=["inventory_check","stock_alert"],
    security=SecurityScheme(scheme_type="apiKey"))

writer_card = AgentCard(
    name="writer_agent",
    description="Writes concise executive summaries for C-suite audiences.",
    endpoint="agent://writer",
    capabilities=["executive_summary","report_writing"],
    security=SecurityScheme(scheme_type="oauth2"))

# The Remote Agent wraps the LangGraph agent built from Lab 1/2 ------------
def make_remote(card, handler_map):
    ra = RemoteAgent(card=card, auth_provider=a2a_auth)
    for cap, fn in handler_map.items():
        ra.register_handler(cap, fn)
    return ra

# Register cards in a discoverable registry
registry = AgentRegistry()
for c in [analytics_card, inventory_card, writer_card]:
    registry.register(c)
print("Registry contents:")
for c in registry.list_all():
    print(f"  {c.name:17s} auth={c.security.scheme_type:7s} "
          f"capabilities={c.capabilities}")


Registry contents:
  analytics_agent   auth=apiKey  capabilities=['kpi_report', 'revenue_query', 'data_analysis']
  inventory_agent   auth=apiKey  capabilities=['inventory_check', 'stock_alert']
  writer_agent      auth=oauth2  capabilities=['executive_summary', 'report_writing']


## Step 3 — User login & dynamic agent instantiation

The supervisor builds each Remote Agent **per session** — the underlying
LangGraph agent is created on demand with the caller's credentials, so every
MCP request is auditable back to the user.


In [3]:
def bind_user(session):
    """Instantiate the three Remote Agents for *this* user session."""
    # --- Analytics (uses Lab 1+2 builder, with Skill on) -----------------
    lg_analytics, _, _ = build_analytics_agent(cred_factory, session,
                                                 apply_skill=True,
                                                 langfuse_handler=lh)
    def h_kpi(data):
        msg = data.get("request","KPI report")
        r = lg_analytics.invoke({"messages":[{"role":"user","content":msg}]})
        return {"report": r["messages"][-1].content}
    analytics_remote = make_remote(analytics_card, {"kpi_report": h_kpi,
                                                      "revenue_query": h_kpi,
                                                      "data_analysis": h_kpi})

    # --- Inventory --------------------------------------------------------
    try:
        lg_inv, _, _ = build_inventory_agent(cred_factory, session,
                                               langfuse_handler=lh)
        def h_inv(data):
            r = lg_inv.invoke({"messages":[{"role":"user",
                 "content":data.get("request","Stock status summary")}]})
            return {"report": r["messages"][-1].content}
        inventory_remote = make_remote(inventory_card,
                                        {"inventory_check": h_inv,
                                         "stock_alert": h_inv})
    except PermissionError:
        inventory_remote = None

    # --- Writer (pure LLM) ------------------------------------------------
    try:
        lg_writer = build_writer_agent(cred_factory, session,
                                         langfuse_handler=lh)
        def h_sum(data):
            # Accept either 'data' (already-produced text) or 'request' (raw user query)
            payload = data.get("data") or data.get("request") or ""
            r = lg_writer.invoke({"messages":[{"role":"user",
                 "content":"Summarise:\n" + payload}]})
            return {"summary": r["messages"][-1].content}
        writer_remote = make_remote(writer_card,
                                     {"executive_summary": h_sum,
                                      "report_writing":  h_sum})
    except PermissionError:
        writer_remote = None

    return {"analytics_agent": analytics_remote,
            "inventory_agent": inventory_remote,
            "writer_agent":    writer_remote}


## Step 4 — Supervisor with auto-discovery

The Supervisor does four things per user request:

1. **Classify** the request with an LLM — pick the capability needed.
2. **Discover** which agents advertise that capability (via the Registry).
3. **Intersect** with the user's grants — drop anything the user cannot use.
4. **Derive credentials** from the session and **delegate** the task via A2A.

It is called "auto" because the supervisor picks and connects on its own —
the user never names an agent.


In [4]:
def classify_intent(request: str) -> str:
    """Choose one capability from the catalogue."""
    catalogue = ["kpi_report", "inventory_check", "executive_summary", "direct"]
    resp = oai.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":
            "Pick exactly one label describing the user request. "
            "Options: " + ", ".join(catalogue)
            + ". Reply with the label only."},
            {"role":"user","content":request}],
    )
    label = (resp.choices[0].message.content or "").strip().lower()
    return label if label in catalogue else "direct"


def run_supervisor(session, request: str,
                     remotes: dict, verbose: bool = True):
    """End-to-end supervisor pass for one user request."""
    cap = classify_intent(request)
    if verbose: print(f"  [supervisor] classified -> {cap}")
    if cap == "direct":
        return f"I can help with KPI reports, inventory status, or exec summaries."

    # Discover who can handle this
    candidates = registry.search(cap)
    allowed_names = cred_factory.available_agents(session)
    # keep only agents this user has a grant for
    candidates = [c for c in candidates if c.name in allowed_names]
    if not candidates:
        return f"Your account has no agent authorised to handle '{cap}'."

    chosen = candidates[0]
    if verbose: print(f"  [supervisor] chosen agent -> {chosen.name} "
                       f"({chosen.security.scheme_type})")

    ra = remotes.get(chosen.name)
    if ra is None:
        return f"Selected agent '{chosen.name}' is not bound for this session."

    # Derive A2A credentials from the session
    creds = cred_factory.derive_a2a_credentials(session, chosen.name,
                                                  chosen.security.scheme_type)
    client = ClientAgent(f"supervisor_for_{session.user_id}")
    client.register_remote(ra)
    client.set_credentials(chosen.endpoint, creds)

    task = client.submit_task(chosen.endpoint, cap, {"request": request})
    if task.status == TaskStatus.COMPLETED:
        out = task.output_data
        return out.get("report") or out.get("summary") or str(out)
    return f"[{task.status.value}] {task.error}"


## Step 5 — Run as `admin_thiem` (full access)


In [5]:
session_admin = idp.login("admin_thiem", "admin456")
remotes_admin = bind_user(session_admin)

for q in [
    "Produce a KPI report: March 2025 vs February 2025, all regions.",
    "Which products are running low on stock?",
    "Write an executive summary about our Q1 2025 sales.",
    "Hello, what can you do?",
]:
    print(f"\n========== {q} ==========")
    print(run_supervisor(session_admin, q, remotes_admin))



========== Produce a KPI report: March 2025 vs February 2025, all regions. ==========


  [supervisor] classified -> kpi_report
  [supervisor] chosen agent -> analytics_agent (apiKey)
### KPI Report: March 2025 vs February 2025 (All Regions)

| Region      | Feb 2025 Revenue | Mar 2025 Revenue | Change (%)            | Status            |
|-------------|---------------------|---------------------|-----------------------|-------------------|
| Hanoi       | 39,960,000 VND      | 50,470,000 VND      | +26.21%               | STRONG GROWTH   |
| HCMC        | 72,960,000 VND      | 62,400,000 VND      | -14.41%               | NEEDS ATTENTION |
| Da Nang     | 30,460,000 VND      | 19,970,000 VND      | -34.41%               | NEEDS ATTENTION |

**Executive Summary:**
Hanoi experienced a significant revenue growth of 26.21%, marking a strong performance. Conversely, HCMC and Da Nang saw revenue declines of 14.41% and 34.41%, respectively, with Da Nang's decrease being particularly notable and requiring attention. Overall, Hanoi was the best-performing region in this compariso

## Step 6 — Same supervisor, different user = different results

`analyst_duc` has `analytics_agent` + `writer_agent` but **no inventory
access**. The supervisor detects this during grant-intersection and returns
a policy-driven refusal. No data leaks, no stack traces.


In [6]:
session_duc = idp.login("analyst_duc", "duc123")
remotes_duc = bind_user(session_duc)

for q in [
    "Produce a KPI report: March 2025 vs February 2025, all regions.",
    "Which products are running low on stock?",
    "Write an executive summary of the KPI report you just produced.",
]:
    print(f"\n========== analyst_duc: {q} ==========")
    print(run_supervisor(session_duc, q, remotes_duc))



========== analyst_duc: Produce a KPI report: March 2025 vs February 2025, all regions. ==========
  [supervisor] classified -> kpi_report
  [supervisor] chosen agent -> analytics_agent (apiKey)
Here is the KPI report for March 2025 vs February 2025 across all regions:

| Region       | Feb 2025 Revenue | Mar 2025 Revenue | Change (%) | Status           |
|--------------|--------------------|--------------------|------------|------------------|
| Hanoi        | 39,960,000 VND     | 50,470,000 VND     | +26.23%    | STRONG GROWTH  |
| HCMC         | 72,960,000 VND     | 62,400,000 VND     | -14.45%    | NEEDS ATTENTION |
| Da Nang      | 30,460,000 VND     | 19,970,000 VND     | -34.42%    | NEEDS ATTENTION |

Summary:
- Hanoi experienced significant growth, with revenue increasing by approximately 26.23%. 
- HCMC's revenue declined by about 14.45%, signaling a need for attention.
- Da Nang saw a substantial decrease in revenue, dropping by roughly 34.42%, which also requires closer mo

## Step 7 — `analyst_mai` (has only `analytics_agent`)


In [7]:
session_mai = idp.login("analyst_mai", "mai123")
remotes_mai = bind_user(session_mai)

for q in [
    "Give me a KPI report for March 2025 vs February 2025.",
    "Write an executive summary.",         # no writer grant
    "What stock is low?",                  # no inventory grant
]:
    print(f"\n========== analyst_mai: {q} ==========")
    print(run_supervisor(session_mai, q, remotes_mai))

flush()



========== analyst_mai: Give me a KPI report for March 2025 vs February 2025. ==========
  [supervisor] classified -> kpi_report
  [supervisor] chosen agent -> analytics_agent (apiKey)
| Region      | Feb 2025 Revenue | Mar 2025 Revenue | % Change             | Status             |
|-------------|------------------|------------------|----------------------|--------------------|
| Hanoi       | 39,960,000 VND   | 50,470,000 VND   | +26.20%              | STRONG GROWTH    |
| HCMC        | 72,960,000 VND   | 62,400,000 VND   | -14.47%              | NEEDS ATTENTION   |
| Da Nang     | 30,460,000 VND   | 19,970,000 VND   | -34.45%              | NEEDS ATTENTION   |

### Executive Summary:
Hanoi experienced a substantial growth of +26.2%, making it the best-performing region this period. HCMC and Da Nang both saw declines exceeding 10%, indicating a need for attention. Overall, Hanoi's boost offset some downward trends in the other regions.

========== analyst_mai: Write an executive summ

## Takeaways

- The supervisor is a thin routing layer. **Everything security-relevant**
  (who may call which agent, with which MCP scopes) flows from the
  SessionToken through the CredentialFactory.
- Each Remote Agent keeps its **own Agent Card + security scheme**, mixing
  API-key and OAuth agents in the same system.
- Adding a new specialist is three steps: build it with the agent_builder,
  write its Agent Card, register it — the supervisor picks it up via
  Registry discovery automatically.
- **Langfuse** traces every LLM call with `user_id` metadata, so you can
  filter the UI by user and audit any session end-to-end.
